# 01. Setup & Baseline (Text Cell)

**Phase 1 → Phase 2 진입**: HuggingFace datasets 로드 → DSC 베이스라인 (clean) → 모델 5종 베이스라인 metric 수집

DSC v5 framework — text × classification (ADR-016) + text × regression (ADR-017) 사전등록.

분류 튜닝 dataset 3종: `fancyzhx/ag_news` / `stanfordnlp/imdb` / `SetFit/20_newsgroups`
회귀 튜닝 dataset 3종: `Yelp/yelp_review_full` (50K) / `mteb/amazon_reviews_multi` 'en' (200K) / `SetFit/sst5`

---


## 0. 환경 설정

Colab T4 GPU 가정. dsc/ 위치는 자동 검색 (G드라이브 어디에 있어도 OK).


In [ ]:
# ============================================================
# 0-1. Drive 마운트 + dsc/ 자동 검색 + sys.path 등록
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json
import numpy as np
import pandas as pd
import torch


def _find_dsc_base():
    """G드라이브에서 dsc_framework/__init__.py가 들어있는 디렉토리 찾기.

    1) 알려진 후보 우선 (빠른 경로)
    2) 깊이 3까지 명시 glob (recursive=True는 큰 Drive에서 느림)
    Returns: 찾은 경로 또는 None.
    """
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    fast = [
        f'{root}/capstone/dsc',
        f'{root}/dsc',
        f'{root}/capstone-dsc',
    ]
    for c in fast:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pattern in [
        f'{root}/*/dsc_framework/__init__.py',
        f'{root}/*/*/dsc_framework/__init__.py',
        f'{root}/*/*/*/dsc_framework/__init__.py',
    ]:
        for hit in glob.glob(pattern):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    drive_root = '/content/drive/MyDrive'
    listing = os.listdir(drive_root) if os.path.isdir(drive_root) else []
    raise RuntimeError(
        'dsc_framework/ 폴더를 G드라이브에서 못 찾음.\n'
        '확인 사항:\n'
        '  1) G드라이브 클라이언트가 sync 완료 상태인지 (commit 직후면 잠시 대기 후 재시도).\n'
        '  2) Drive 마운트가 됐는지 — `!ls /content/drive/MyDrive` 출력 확인.\n'
        f'  현재 /content/drive/MyDrive 안 내용: {listing[:20]}\n'
        '  3) 위 1~2 모두 OK인데도 실패면 사용자 Drive에 dsc_framework 폴더 자체가 없음 → push/sync 재확인.'
    )

RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/text'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BASE: {BASE}')
fw_contents = sorted(f for f in os.listdir(f'{BASE}/dsc_framework') if not f.startswith('_'))
print(f'  dsc_framework 폴더 ({len(fw_contents)}개): {fw_contents}')

REQUIRED_FILES = [
    'shared_metrics.py', 'classification_cell.py', 'regression_cell.py',
    'image_cell.py', 'text_cell.py', 'text_cell_regression.py',
    'column_detection.py', 'data_type_detection.py', 'router.py',
    'text_polluters', 'image_polluters',
]
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'{BASE}/dsc_framework/{f}')]
if missing:
    raise RuntimeError(
        f'dsc_framework/ 폴더는 있지만 다음 파일들이 sync 안 됨: {missing}\n'
        '→ G드라이브 클라이언트에서 sync 완료될 때까지 대기 후 재실행.\n'
        '→ 강제 sync: G드라이브 폴더 열고 새로고침 또는 클라이언트 재시작.\n'
        '→ Colab Drive view가 stale일 수 있음: drive.flush_and_unmount() 후 재마운트.'
    )
print(f'device: {device}, torch: {torch.__version__}')


In [ ]:
# ============================================================
# 0-2. 의존성 설치 (Colab 환경) — 패키지 설치 후 다음 셀부터 import
# ============================================================
%pip install -q transformers>=4.30 datasets>=2.10 xgboost>=1.7


In [ ]:
# ============================================================
# 0-3. dsc_framework / datasets import — 위 셀 실행 완료 후
# ============================================================
import warnings
warnings.simplefilter('default')

from datasets import load_dataset

# dsc_framework는 robust __init__.py라 일부 cell 누락 시 warning만 발생.
# text_cell이 진짜 못 import되면 ModuleNotFoundError raise — 그 경우 진단.
try:
    from dsc_framework.text_cell import compute_dsc_text
    from dsc_framework.text_cell_regression import compute_dsc_text_regression
except ModuleNotFoundError as e:
    fw = f'{BASE}/dsc_framework'
    have = sorted(os.listdir(fw)) if os.path.isdir(fw) else []
    raise RuntimeError(
        f'text_cell import 실패: {e}\n'
        f'dsc_framework/ 안 파일: {have}\n'
        '→ shared_metrics.py / text_cell.py 등 의존 파일이 Drive에 있는지 확인. '
        'sync 미완료면 잠시 대기 후 재실행.'
    ) from e

print('dsc_framework import OK.')
print('compute_dsc_text:', compute_dsc_text)
print('compute_dsc_text_regression:', compute_dsc_text_regression)


## 1. 튜닝 dataset 로드

ADR-016 §3-1 / ADR-017 §3-1 freeze.


In [ ]:
# 분류 트랙
ag_news = load_dataset('fancyzhx/ag_news')
imdb    = load_dataset('stanfordnlp/imdb')
news20  = load_dataset('SetFit/20_newsgroups')

# 회귀 트랙 (Yelp/Amazon은 sample_cap 적용)
yelp    = load_dataset('Yelp/yelp_review_full')
amazon  = load_dataset('mteb/amazon_reviews_multi', 'en')
sst5    = load_dataset('SetFit/sst5')

print('sizes:',
    {'ag_news': len(ag_news['train']), 'imdb': len(imdb['train']),
     '20news': len(news20['train']), 'yelp': len(yelp['train']),
     'amazon': len(amazon['train']), 'sst5': len(sst5['train'])})


## 2. 샘플링 (ADR-017 §4 freeze)

Yelp / Amazon은 stratified random_state=42, train 50K (Yelp) / 200K (Amazon) / test 5K.


In [ ]:
def stratified_sample(ds, label_key, n_per_split, seed=42):
    rng = np.random.RandomState(seed)
    df = ds.to_pandas()
    parts = [g.sample(min(len(g), n_per_split // df[label_key].nunique()), random_state=seed)
             for _, g in df.groupby(label_key)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

yelp_train_s  = stratified_sample(yelp['train'],   'label',  50000)
yelp_test_s   = stratified_sample(yelp['test'],    'label',   5000)
amazon_train_s= stratified_sample(amazon['train'], 'label', 200000)
amazon_test_s = stratified_sample(amazon['test'],  'label',   5000)
print('Yelp:', len(yelp_train_s), len(yelp_test_s), '| Amazon:', len(amazon_train_s), len(amazon_test_s))


## 3. DSC 베이스라인 (clean DSC, default 가중치)

ADR-015 원칙: 본 단계의 default DSC는 합격선 출발점 측정. 운영 가중치는 Phase 4 LLM 호출 결과 사용.


In [ ]:
def df_to_text_label(ds, text_key='text', label_key='label'):
    if isinstance(ds, pd.DataFrame):
        return ds[text_key].tolist(), ds[label_key].tolist()
    return ds[text_key], ds[label_key]

# 분류 cell — clean baseline
for name, ds in [('ag_news', ag_news['train']),
                  ('imdb',   imdb['train']),
                  ('20news', news20['train'])]:
    texts, labels = df_to_text_label(ds)
    r = compute_dsc_text(texts[:5000], labels[:5000],
                         use_embeddings=True, sample_cap=1000, random_state=42)
    print(f"{name:8s}  DSC={r['score']}  grade={r['grade']}")

# 회귀 cell — clean baseline (target=label cast to float)
for name, ds in [('yelp_50k',   yelp_train_s),
                  ('amazon_200k', amazon_train_s),
                  ('sst5',       pd.DataFrame(sst5['train']))]:
    texts, lbs = df_to_text_label(ds)
    targets = [float(x) for x in lbs]
    r = compute_dsc_text_regression(texts[:5000], targets[:5000],
                                    use_embeddings=True, sample_cap=1000, random_state=42)
    print(f"{name:12s} DSC={r['score']}  grade={r['grade']}")


## 4. 모델 베이스라인 — clean 학습

5 모델 × 6 dataset baseline accuracy/R² 수집. 학습 시간 절약 위해 각 dataset의 train_sub(예: 5000)로 sanity. Phase 2 정식 학습은 03 노트북에서 풀 train으로 재실행.

다음 셀들은 GPU 환경에서 실행 — 본 노트북 stub은 함수 정의만 둠.


In [ ]:
# Transformer head 분류 학습 (DistilBERT/BERT/RoBERTa 공유)
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# 함수 정의는 03_training_text.ipynb에 풀 버전 작성


In [ ]:
# TextCNN 분류 (random init embedding + 3-kernel conv)
# class TextCNN(torch.nn.Module): ...  # 03_training_text.ipynb 참조


In [ ]:
# LogReg + TF-IDF baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def logreg_tfidf_accuracy(train_texts, train_labels, test_texts, test_labels,
                          max_features=20000, ngram_range=(1, 2)):
    vec = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    Xtr = vec.fit_transform(train_texts)
    Xte = vec.transform(test_texts)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, random_state=42).fit(Xtr, train_labels)
    p = clf.predict(Xte)
    return accuracy_score(test_labels, p), f1_score(test_labels, p, average='macro')


---

다음: `02_pollution_and_dsc_text.ipynb` — 7 polluter × 6 level × 6 dataset 스윕.
